# TP 1 — Apprentissage supervise 2A (correction R)

**Contenu du sujet :**
- **Partie 1** : regle de Bayes et risque de Bayes en classification binaire (TD).
- **Partie 2** : simulation de donnees, jeux reels, algorithme des *k* plus proches voisins (k-NN).

> Les sujets sont rediges pour **R**. Cette correction utilise donc directement les packages R :
> `class` (k-NN), `caret` (validation croisee), `mlbench` (jeu `Vehicle`), et **plotly** pour les graphiques.
> Elle reproduit fidelement la correction Python fournie en parallele (memes calculs, memes conclusions).


## Brief de demarrage — Les outils de ce TP

> **A lire avant de commencer.** Ce premier TP pose les **fondations** du cours : la reference theorique (la
> regle de Bayes) et le premier vrai classifieur (le k-NN). Voici comment ils fonctionnent.

**1. Regle de Bayes et risque de Bayes — la reference *optimale*.**
En classification binaire avec le cout 0-1, la **meilleure decision possible** consiste a predire la classe la
plus probable :
$$g^\star(x) = \mathbb{1}\{\eta(x) \ge 1/2\}, \qquad \eta(x) = \mathbb{P}(Y = 1 \mid X = x).$$
Son taux d'erreur, le **risque de Bayes** $L^\star = \mathbb{E}[\min(\eta(X), 1-\eta(X))]$, est le **plancher**
que **aucun** modele ne peut battre. On ne l'atteint jamais en pratique (on ne connait pas $\eta$), mais il sert
de **point de comparaison** : un bon modele s'en approche.

**2. k plus proches voisins (k-NN) — estimer $\eta$ *localement*.**
Idee : $\eta(x)$ est estime par la **proportion de voisins** de la classe 1 parmi les $k$ points les plus
proches de $x$. On classe ensuite au vote majoritaire.
- **$k$ petit** (ex. 1) : colle aux donnees, frontiere tres irreguliere ⇒ **variance elevee** (surajustement) ;
- **$k$ grand** : moyenne sur un large voisinage, frontiere lisse ⇒ **biais eleve** (sous-ajustement).
- Le k-NN mesure des **distances** : les variables doivent etre a la **meme echelle** (standardisation).

Le bon $k$ realise le meilleur **compromis biais-variance** ; on le choisit en comparant l'erreur sur un jeu de
test (ou par validation croisee).

> **Methode du TP.** (Partie 1) calculer la regle et le risque de Bayes sur un modele connu ; (Partie 2)
> simuler des donnees puis appliquer le k-NN, faire **varier $k$** et observer son effet sur l'erreur.

In [1]:
source(if (file.exists("plot_theme.R")) "plot_theme.R" else "../plot_theme.R")  # theme + charger_packages()
charger_packages(c("mlbench",   # jeu Vehicle
                   "class",     # knn
                   "caret",     # validation croisee, decoupage
                   "plotly"))   # visualisations

SEED <- 42
set.seed(SEED)   # graine unique posee une fois : tout le TP est reproductible

## Partie 1 — Regle de Bayes et risque en classification binaire

### Rappel de cours

On observe un couple $(X, Y)$ ou $Y \in \{0, 1\}$ (classification **binaire**). Un **classifieur** est une
fonction $g : \mathcal{X} \to \{0,1\}$. On mesure sa qualite par le **risque** (probabilite d'erreur) associe au
cout 0-1 :
$$L(g) = \mathbb{P}\big(g(X) \neq Y\big).$$

L'objet central est la **fonction de regression** (probabilite a posteriori) :
$$\eta(x) = \mathbb{P}(Y = 1 \mid X = x).$$

Le classifieur qui **minimise** le risque est le **classifieur de Bayes** :
$$g^\star(x) = \mathbb{1}\big\{\eta(x) \ge \tfrac{1}{2}\big\}.$$

Son risque, le **risque de Bayes**, est le plus petit risque atteignable :
$$L^\star = \mathbb{E}\big[\min\big(\eta(X),\, 1 - \eta(X)\big)\big].$$

**Intuition.** En un point $x$, la meilleure decision se trompe avec probabilite $\min(\eta(x), 1-\eta(x))$.
On moyenne ensuite sur la loi de $X$.


### Cas 1

Modele : $X_i \sim \mathcal{N}(0,1)$, $U_i \sim \mathcal{U}[0,1]$ independants, et
$$Y_i = \begin{cases} \mathbb{1}(U_i \le 0.1) & \text{si } X_i \le 0 \\ \mathbb{1}(U_i > 0.2) & \text{si } X_i > 0. \end{cases}$$

**Etape 1 — Calcul de $\eta$.** Comme $U$ est independant de $X$ :
- si $x \le 0$ : $\eta(x) = \mathbb{P}(U \le 0.1) = 0.1$ ;
- si $x > 0$ : $\eta(x) = \mathbb{P}(U > 0.2) = 0.8$.

**Etape 2 — Regle de Bayes** : $x \le 0 \Rightarrow g^\star = 0$ ; $x > 0 \Rightarrow g^\star = 1$.
Donc $g^\star(x) = \mathbb{1}\{x > 0\}$.

**Etape 3 — Risque de Bayes.** Comme $\mathbb{P}(X \le 0) = \mathbb{P}(X > 0) = 0.5$ :
$$L^\star = 0.5 \cdot \min(0.1, 0.9) + 0.5 \cdot \min(0.8, 0.2) = 0.5(0.1) + 0.5(0.2) = \boxed{0.15}.$$


### Cas 2

Memes hypotheses avec les seuils $0.2$ et $0.4$ :
- $x \le 0$ : $\eta(x) = 0.2$ ; $x > 0$ : $\eta(x) = \mathbb{P}(U > 0.4) = 0.6$.
- Regle de Bayes : $g^\star(x) = \mathbb{1}\{x > 0\}$ (identique au Cas 1).
- Risque de Bayes :
$$L^\star = 0.5 \cdot \min(0.2, 0.8) + 0.5 \cdot \min(0.6, 0.4) = 0.5(0.2) + 0.5(0.4) = \boxed{0.30}.$$


### Cas 3 (donnees reelles « cafe »)

Ici $X$ = sexe (femme / homme) et $Y$ = « cafe sucre » (1) ou non (0). L'enonce donne :
$$\mathbb{P}(Y=1 \mid \text{femme}) = 0.2, \qquad \mathbb{P}(Y=1 \mid \text{homme}) = 0.1.$$

**Regle de Bayes.** Les deux probabilites sont $< 0.5$ : quel que soit le sexe, on predit « non sucre ».
Donc $g^\star(x) = 0$ pour tout $x$.

**Risque de Bayes.** Il vaut $L^\star = \mathbb{P}(\text{femme})\,(0.2) + \mathbb{P}(\text{homme})\,(0.1)$.
On **ne peut pas** donner de valeur numerique : l'enonce ne fournit **pas** la repartition femmes/hommes.
Si l'echantillon etait equilibre ($50\%/50\%$), on aurait $L^\star = 0.5(0.2) + 0.5(0.1) = 0.15$.


### Verification numerique par simulation (Monte-Carlo)

On simule un grand echantillon et on estime empiriquement le risque de Bayes en appliquant la regle
$g^\star(x) = \mathbb{1}\{x > 0\}$. On doit retrouver $\approx 0.15$ (Cas 1) et $\approx 0.30$ (Cas 2).


In [2]:
simule_cas <- function(n, a, b) {
  # Modele du sujet : Y = 1(U<=a) si X<=0, sinon 1(U>b)
  X <- rnorm(n)
  U <- runif(n)
  Y <- ifelse(X <= 0, as.integer(U <= a), as.integer(U > b))
  data.frame(X = X, Y = Y)
}

risque_bayes_empirique <- function(n, a, b) {
  d <- simule_cas(n, a, b)
  y_pred <- as.integer(d$X > 0)   # regle de Bayes g*(x) = 1{x > 0}
  mean(y_pred != d$Y)
}

N <- 500000
for (cas in list(list("Cas 1", 0.1, 0.2, 0.15), list("Cas 2", 0.2, 0.4, 0.30)))
  cat(sprintf("%s : risque empirique = %.4f  (theorique %.2f)\n",
              cas[[1]], risque_bayes_empirique(N, cas[[2]], cas[[3]]), cas[[4]]))

Cas 1 : risque empirique = 0.1500  (theorique 0.15)
Cas 2 : risque empirique = 0.3006  (theorique 0.30)


### Question 2 — Indicateur de complexite et classement

Le **risque de Bayes $L^\star$** est l'indicateur naturel de difficulte : c'est l'erreur **incompressible**.
Plus $L^\star$ est grand, plus le probleme est difficile. Un probleme est **facile** quand $\eta(x)$ est
**loin de $1/2$** (classes bien separees), et **difficile** quand $\eta(x)$ est **proche de $1/2$**.

**Classement (du plus facile au plus difficile) :**
$$\underbrace{\text{Cas 1}}_{L^\star = 0.15} \;<\; \underbrace{\text{Cas 3}}_{L^\star = 0.15 \text{ si equilibre}} \;<\; \underbrace{\text{Cas 2}}_{L^\star = 0.30}.$$


## Partie 2 — Manipulation de donnees et algorithme des k plus proches voisins

### Rappel de cours — Regle de decision du k-NN

Pour classer un nouveau point $x$, l'algorithme des **$k$ plus proches voisins** :
1. calcule la distance (souvent **euclidienne**) de $x$ a tous les points d'apprentissage ;
2. retient les **$k$** points les plus proches ;
3. predit la classe **majoritaire** parmi ces $k$ voisins (**vote**).

C'est une methode **non parametrique**. Deux points de vigilance :
- le choix de **$k$** contröle le compromis biais-variance ($k$ petit ⇒ surajustement ; $k$ grand ⇒ sous-ajustement) ;
- la methode est **sensible a l'echelle** des variables ⇒ il faut **standardiser**.


### Exercice 1 — Situation du Cas 2

**1.1 & 1.2** On simule un echantillon d'**apprentissage** de taille $1000$ et un echantillon **independant**
de taille $200$. Ce second echantillon sert de **jeu de test** : il estime l'erreur de **generalisation**
(sur des donnees jamais vues) sans biais optimiste.


In [3]:
# 1.1 apprentissage (n = 1000) et 1.2 test (n = 200), Cas 2 : a = 0.2, b = 0.4
tr <- simule_cas(1000, 0.2, 0.4)
te <- simule_cas(200,  0.2, 0.4)
cat("Apprentissage :", nrow(tr), "obs | Test :", nrow(te), "obs\n")

Apprentissage : 1000 obs | Test : 200 obs


**2 & 3** On entraine un k-NN **sans reglage** (valeur par defaut $k = 5$) et on evalue l'erreur sur le test.
Le risque de Bayes du Cas 2 est $0.30$ : c'est la borne inferieure atteignable.


In [4]:
# k-NN sans reglage : k = 5 par defaut
pred_tr <- knn(train = tr["X"], test = tr["X"], cl = factor(tr$Y), k = 5)
pred_te <- knn(train = tr["X"], test = te["X"], cl = factor(tr$Y), k = 5)

cat(sprintf("Erreur d'apprentissage : %.3f\n", mean(pred_tr != tr$Y)))
cat(sprintf("Erreur de test         : %.3f   (risque de Bayes = 0.30)\n", mean(pred_te != te$Y)))

Erreur d'apprentissage : 0.232


Erreur de test         : 0.355   (risque de Bayes = 0.30)


**Lecture.** L'erreur de test est du **même ordre que le risque de Bayes** $L^\star = 0.30$, en général un peu
au-dessus, et ne peut pas descendre durablement en dessous. Trois remarques :
- avec $k = 5$, l'estimation locale de $\eta$ repose sur 5 votes seulement : elle est **bruitée**, d'où un
  **excès de risque** par rapport à la règle de Bayes ;
- avec seulement $200$ points de test, l'erreur estimée **fluctue** : son écart-type vaut environ
  $\sqrt{0.3 \times 0.7 / 200} \approx 0.03$. Un écart de quelques points à $0.30$ n'est donc pas significatif ;
- l'erreur d'apprentissage est plus **faible** (le modèle « voit » ces points, et chaque point est son propre
  voisin) : il ne faut **jamais** évaluer un modèle sur ses données d'entraînement.

**Effet de $k$.** Le graphique suivant fait varier $k$ : pour $k = 1$, l'erreur d'apprentissage est nulle mais
l'erreur de test est forte (théorème de Cover–Hart : asymptotiquement, $L_{1\text{-NN}} \le 2L^\star(1-L^\star)
= 0.42$) ; quand $k$ augmente, l'estimation locale de $\eta$ se stabilise et l'erreur de test se rapproche de
$L^\star = 0.30$. Si $k$ devient **trop grand**, le voisinage déborde de part et d'autre de $x = 0$ et l'erreur
remonte près de la frontière (biais).

In [5]:
ks <- seq(1, 201, by = 4)
erreurs <- t(sapply(ks, function(k) c(
  apprentissage = mean(knn(tr["X"], tr["X"], factor(tr$Y), k = k) != tr$Y),
  test          = mean(knn(tr["X"], te["X"], factor(tr$Y), k = k) != te$Y)
)))

plot_ly(x = ks) %>%
  add_trace(y = erreurs[, "apprentissage"], type = "scatter", mode = "lines+markers",
            name = "Erreur apprentissage", line = list(color = pick_color(0)), marker = list(color = pick_color(0))) %>%
  add_trace(y = erreurs[, "test"], type = "scatter", mode = "lines+markers",
            name = "Erreur test", line = list(color = pick_color(1)), marker = list(color = pick_color(1))) %>%
  add_lines(y = rep(0.30, length(ks)), name = "Risque de Bayes = 0.30",
            line = list(color = HIGHLIGHT, dash = "dash")) %>%
  layout(title = "Cas 2 : erreur du k-NN en fonction de k",
         xaxis = list(title = "k (nombre de voisins)"), yaxis = list(title = "Taux d'erreur")) %>%
  apply_dark_theme()

HTML widgets cannot be represented in plain text (need html)

### Exercice 2 — Jeu de donnees reel « Vehicle »

Le sujet utilise `mlbench::Vehicle` : reconnaitre le **type de vehicule** (4 classes) a partir de variables de
forme extraites de silhouettes.

**1.1 & 1.2** Chargement et description.


In [6]:
data(Vehicle)
X_all <- Vehicle[, setdiff(names(Vehicle), "Class")]
y <- Vehicle$Class

cat("Dimensions :", dim(X_all), "\n")
cat("Classes    :", paste(levels(y), collapse = ", "), "\n")
head(Vehicle[, 1:6])


Dimensions : 846 18 


Classes    : bus, opel, saab, van 


,Comp,Circ,D.Circ,Rad.Ra,Pr.Axis.Ra,Max.L.Ra
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,95,48,83,178,72,10
2,91,41,84,141,57,9
3,104,50,106,209,66,10
4,93,41,82,159,63,9
5,85,44,70,205,103,52
6,107,57,106,172,50,6


In [7]:

cc <- as.data.frame(table(y)); names(cc) <- c("Classe", "Effectif")
plot_ly(cc, x = ~Classe, y = ~Effectif, type = "bar",
        text = ~Effectif, textposition = "outside",
        marker = list(color = pick_color(0))) %>%
  layout(title = "Distribution des classes du jeu Vehicle",
         xaxis = list(title = "Classe"), yaxis = list(title = "Effectif")) %>%
  apply_dark_theme()


HTML widgets cannot be represented in plain text (need html)

In [8]:

# Description du jeu de donnees : structure et resume statistique
str(Vehicle)
summary(X_all[, 1:6])


'data.frame':	846 obs. of  19 variables:
 $ Comp        : num  95 91 104 93 85 107 97 90 86 93 ...
 $ Circ        : num  48 41 50 41 44 57 43 43 34 44 ...
 $ D.Circ      : num  83 84 106 82 70 106 73 66 62 98 ...
 $ Rad.Ra      : num  178 141 209 159 205 172 173 157 140 197 ...
 $ Pr.Axis.Ra  : num  72 57 66 63 103 50 65 65 61 62 ...
 $ Max.L.Ra    : num  10 9 10 9 52 6 6 9 7 11 ...
 $ Scat.Ra     : num  162 149 207 144 149 255 153 137 122 183 ...
 $ Elong       : num  42 45 32 46 45 26 42 48 54 36 ...
 $ Pr.Axis.Rect: num  20 19 23 19 19 28 19 18 17 22 ...
 $ Max.L.Rect  : num  159 143 158 143 144 169 143 146 127 146 ...
 $ Sc.Var.Maxis: num  176 170 223 160 241 280 176 162 141 202 ...
 $ Sc.Var.maxis: num  379 330 635 309 325 957 361 281 223 505 ...
 $ Ra.Gyr      : num  184 158 220 127 188 264 172 164 112 152 ...
 $ Skew.Maxis  : num  70 72 73 63 127 85 66 67 64 64 ...
 $ Skew.maxis  : num  6 9 14 6 9 5 13 3 2 4 ...
 $ Kurt.maxis  : num  16 14 9 10 11 9 1 3 14 14 ...
 $ Kurt.Maxis  

      Comp             Circ           D.Circ           Rad.Ra     
 Min.   : 73.00   Min.   :33.00   Min.   : 40.00   Min.   :104.0  
 1st Qu.: 87.00   1st Qu.:40.00   1st Qu.: 70.00   1st Qu.:141.0  
 Median : 93.00   Median :44.00   Median : 80.00   Median :167.0  
 Mean   : 93.68   Mean   :44.86   Mean   : 82.09   Mean   :168.9  
 3rd Qu.:100.00   3rd Qu.:49.00   3rd Qu.: 98.00   3rd Qu.:195.0  
 Max.   :119.00   Max.   :59.00   Max.   :112.00   Max.   :333.0  
   Pr.Axis.Ra        Max.L.Ra     
 Min.   : 47.00   Min.   : 2.000  
 1st Qu.: 57.00   1st Qu.: 7.000  
 Median : 61.00   Median : 8.000  
 Mean   : 61.69   Mean   : 8.567  
 3rd Qu.: 65.00   3rd Qu.:10.000  
 Max.   :138.00   Max.   :55.000  

**1.3 Combien de predicteurs potentiels ?** Le jeu contient **18** variables de forme (toutes les colonnes
sauf `Class`).

**1.4 Selection** : on retire les variables dont le nom commence par « Skew » ou « Kurt » (asymetrie /
aplatissement), sans tenir compte de la casse. Il en reste **14**.


In [9]:

cat("Nombre de predicteurs potentiels :", ncol(X_all), "\n")

a_exclure <- grep("^(Skew|Kurt)", names(X_all), ignore.case = TRUE, value = TRUE)
X_sel <- X_all[, setdiff(names(X_all), a_exclure)]

cat("Variables exclues     :", paste(a_exclure, collapse = ", "), "\n")
cat("Predicteurs conserves :", ncol(X_sel), "->", paste(names(X_sel), collapse = ", "), "\n")


Nombre de predicteurs potentiels : 18 


Variables exclues     : Skew.Maxis, Skew.maxis, Kurt.maxis, Kurt.Maxis 


Predicteurs conserves : 14 -> Comp, Circ, D.Circ, Rad.Ra, Pr.Axis.Ra, Max.L.Ra, Scat.Ra, Elong, Pr.Axis.Rect, Max.L.Rect, Sc.Var.Maxis, Sc.Var.maxis, Ra.Gyr, Holl.Ra 


In [10]:

# Distribution des variables descriptives conservees
vars <- names(X_sel)
n_cols <- 3
n_rows <- ceiling(length(vars) / n_cols)
plots <- lapply(seq_along(vars), function(i) {
  v <- vars[i]
  plot_ly(x = X_sel[[v]], type = "histogram", nbinsx = 20,
          marker = list(color = pick_color(i - 1)), name = v) %>%
    layout(xaxis = list(title = v), yaxis = list(title = "Effectif"))
})
subplot(plots, nrows = n_rows, titleX = TRUE, margin = 0.05) %>%
  layout(title = "Distribution des variables descriptives conservees (Vehicle)",
         showlegend = FALSE) %>%
  apply_dark_theme(height = 250 * n_rows)


HTML widgets cannot be represented in plain text (need html)

**2. Fonction de cout.** Probleme de classification (4 categories) ⇒ **cout 0-1**, c.-a-d. le **taux d'erreur
de classification** (de facon equivalente, on maximise l'*accuracy*).

**3.1 Premiere etape obligatoire.** Toujours **separer les donnees en apprentissage / test** (ici de facon
stratifiee via `createDataPartition`, qui conserve les proportions de classes). Le test ne doit **jamais**
servir a l'entrainement ni au reglage.


In [11]:
idx <- createDataPartition(y, p = 0.75, list = FALSE)
X_train <- X_sel[idx, ];  X_test <- X_sel[-idx, ]
y_train <- y[idx];        y_test <- y[-idx]
cat("Apprentissage :", nrow(X_train), "| Test :", nrow(X_test), "\n")

Apprentissage : 636 | Test : 210 


**3.2 Sensibilite du k-NN & transformation.** Le k-NN repose sur des **distances** : une variable de grande
amplitude ecrase les autres. Les variables de `Vehicle` ont des **echelles tres differentes** ⇒ on
**standardise** (centrer-reduire). Avec `caret`, le `preProcess = c("center","scale")` est appris **uniquement**
sur le pli d'apprentissage a chaque etape de validation croisee (pas de fuite de donnees).

**3.3 Hyperparametre a regler : $k$.** On l'optimise par **validation croisee** (5 plis) sur une grille de
valeurs impaires (pour eviter les egalites de vote). On retient le $k$ qui maximise l'accuracy, puis on mesure
l'erreur sur le **test**.

**Pourquoi pas $k=4$ car il y a 4 classes ?** Le **nombre de classes** et le **nombre de voisins** sont deux
choses differentes. $k$ contröle le compromis biais-variance ; en multi-classes il n'a aucune raison d'egaler
le nombre de classes. On prend souvent un $k$ **impair** pour limiter les ex-aequo.


In [12]:
ctrl <- trainControl(method = "cv", number = 5)
# Le pretraitement center/scale est appris sur chaque pli d'apprentissage (pas de fuite)
fit <- train(X_train, y_train, method = "knn",
             preProcess = c("center", "scale"),
             tuneGrid = data.frame(k = seq(1, 30, 2)),
             trControl = ctrl)

k_opt <- fit$bestTune$k
cat("Meilleur k (validation croisee) :", k_opt, "\n")
cat(sprintf("Accuracy CV au meilleur k       : %.3f\n", max(fit$results$Accuracy)))

Meilleur k (validation croisee) : 5 


Accuracy CV au meilleur k       : 0.753


In [13]:
res <- fit$results
plot_ly(res, x = ~k, y = ~Accuracy) %>%
  add_trace(type = "scatter", mode = "lines+markers", name = "Accuracy CV",
            line = list(width = 3, color = pick_color(0)), marker = list(size = 8, color = pick_color(2))) %>%
  add_segments(x = k_opt, xend = k_opt, y = min(res$Accuracy), yend = max(res$Accuracy),
               line = list(dash = "dash", color = pick_color(1)), name = paste("k optimal =", k_opt)) %>%
  layout(title = "Reglage de k par validation croisee - Vehicle",
         xaxis = list(title = "k (nombre de voisins)"),
         yaxis = list(title = "Accuracy (validation croisee)")) %>%
  apply_dark_theme()

HTML widgets cannot be represented in plain text (need html)

In [14]:

pred <- predict(fit, X_test)
err_test_vehicle <- mean(pred != y_test)
cat(sprintf("Erreur de prediction (test) avec k = %d : %.3f\n", k_opt, err_test_vehicle))


Erreur de prediction (test) avec k = 5 : 0.262


**Conclusion de l'Exercice 2.**
- La **standardisation** est indispensable pour le k-NN.
- Le **$k$ optimal** est choisi par validation croisee (meilleur compromis biais-variance), independamment du test.
- L'erreur finale est mesuree **une seule fois** sur le test : estimation honnete de la generalisation.


---

## Annexe A — Exemple visuel simple pour comprendre le k-NN

**1. Le vote des voisins.** On construit un petit nuage de 10 points sur 2 axes, on ajoute un nouveau point
(etoile) et on le classe par k-NN (vote majoritaire des $k$ voisins).

**2. Pourquoi standardiser ?** Un cas concret (clients decrits par leur age et leur revenu) montre que le plus
proche voisin **change** selon que l'on standardise ou non les variables.

In [15]:
# 1) Exemple visuel : 10 points, un nouveau point, k = 3
X_demo <- matrix(c(1.0,1.1, 1.4,1.3, 1.1,1.8, 0.7,1.5, 1.8,0.9,
                   4.3,3.8, 3.9,4.4, 4.6,4.2, 3.7,3.6, 4.2,4.8),
                 ncol = 2, byrow = TRUE)
y_demo <- rep(0:1, each = 5)
x_new  <- c(2.8, 2.9)
k <- 3

voisins <- order(sqrt(colSums((t(X_demo) - x_new)^2)))[1:k]
pred <- knn(X_demo, matrix(x_new, 1), factor(y_demo), k = k)

fig <- plot_ly()
for (i in voisins) {   # segments vers les k voisins
  fig <- add_trace(fig, x = c(x_new[1], X_demo[i, 1]), y = c(x_new[2], X_demo[i, 2]),
                   type = "scatter", mode = "lines", showlegend = FALSE,
                   line = list(color = MUTED, dash = "dash"))
}
for (c in 0:1) {
  fig <- add_trace(fig, x = X_demo[y_demo == c, 1], y = X_demo[y_demo == c, 2],
                   type = "scatter", mode = "markers", name = paste("Classe", c),
                   marker = list(size = 12, color = pick_class_color(c)))
}
fig %>%
  add_trace(x = X_demo[voisins, 1], y = X_demo[voisins, 2], type = "scatter", mode = "markers",
            name = paste(k, "plus proches voisins"),
            marker = list(size = 22, color = "rgba(0,0,0,0)", line = list(width = 2, color = HIGHLIGHT))) %>%
  add_trace(x = x_new[1], y = x_new[2], type = "scatter", mode = "markers", name = "Nouveau point",
            marker = list(size = 18, symbol = "star", color = pick_class_color("nouveau"))) %>%
  layout(title = paste0("k-NN simple : k=", k, ", prediction = classe ", pred),
         xaxis = list(title = "Axe 1"), yaxis = list(title = "Axe 2")) %>%
  apply_dark_theme()

HTML widgets cannot be represented in plain text (need html)

In [16]:
# 2) Pourquoi standardiser ? Clients decrits par (age, revenu annuel)
X_clients <- matrix(c(25, 25000, 30, 30000, 65, 61000, 70, 66000), ncol = 2, byrow = TRUE)
noms <- c("A (jeune)", "B (jeune)", "C (senior)", "D (senior)")
x_client <- c(27, 60000)   # client jeune, mais revenu eleve

mu <- colMeans(X_clients); sdv <- apply(X_clients, 2, sd)
espaces <- list(
  "Echelle brute : le revenu domine"               = list(X = X_clients, q = x_client, suffixe = ""),
  "Apres standardisation : age et revenu comptent" = list(X = scale(X_clients), q = (x_client - mu) / sdv,
                                                          suffixe = " (standardise)")
)
couleurs <- rep(c(pick_class_color(0), pick_class_color(1)), each = 2)

panneaux <- lapply(names(espaces), function(titre) {
  e <- espaces[[titre]]
  i <- which.min(sqrt(colSums((t(e$X) - e$q)^2)))
  cat(sprintf("%-48s -> plus proche voisin : %s\n", titre, noms[i]))
  plot_ly() %>%
    add_trace(x = c(e$q[1], e$X[i, 1]), y = c(e$q[2], e$X[i, 2]), type = "scatter", mode = "lines",
              line = list(color = HIGHLIGHT, dash = "dash"), showlegend = FALSE) %>%
    add_trace(x = e$X[, 1], y = e$X[, 2], type = "scatter", mode = "markers+text", text = noms,
              textposition = "top center", showlegend = FALSE,
              marker = list(size = 14, color = couleurs)) %>%
    add_trace(x = e$q[1], y = e$q[2], type = "scatter", mode = "markers", name = "Nouveau client",
              showlegend = titre == names(espaces)[1],
              marker = list(size = 20, symbol = "star", color = pick_class_color("nouveau"))) %>%
    layout(xaxis = list(title = paste0("Age", e$suffixe)), yaxis = list(title = paste0("Revenu", e$suffixe)))
})
subplot(panneaux, nrows = 1, titleX = TRUE, titleY = TRUE, margin = 0.07) %>%
  layout(title = "Le plus proche voisin change apres standardisation") %>%
  apply_dark_theme()

Echelle brute : le revenu domine                 -> plus proche voisin : C (senior)
Apres standardisation : age et revenu comptent   -> plus proche voisin : B (jeune)


HTML widgets cannot be represented in plain text (need html)

---

# 📎 ANNEXE B — Preuve de la formule du risque de Bayes

---

**Objectif.** Montrer rigoureusement que
$$L^\star = L(g^\star) = \mathbb{E}\big[\min\big(\eta(X),\, 1 - \eta(X)\big)\big],$$
et que $g^\star(x) = \mathbb{1}\{\eta(x) \ge \tfrac{1}{2}\}$ est bien **optimal** (aucun classifieur ne fait mieux).

### Étape 1 — Conditionnement sur $X$

Pour un classifieur **quelconque** $g$, le risque 0–1 s'écrit, par la formule de l'espérance totale :
$$L(g) = \mathbb{P}\big(g(X) \neq Y\big) = \mathbb{E}\Big[\, \mathbb{1}\{g(X) \neq Y\} \,\Big]
= \mathbb{E}\Big[\, \underbrace{\mathbb{P}\big(g(X) \neq Y \mid X\big)}_{\text{erreur locale en } X} \,\Big].$$
Il suffit donc de **minimiser l'erreur en chaque point** $x$, puis de moyenner sur la loi de $X$.

### Étape 2 — Erreur locale en un point $x$

En un point $x$ fixé, la prédiction $g(x) \in \{0,1\}$ est une constante. Comme
$\eta(x) = \mathbb{P}(Y = 1 \mid X = x)$ :

$$\mathbb{P}\big(g(X) \neq Y \mid X = x\big) =
\begin{cases}
\mathbb{P}(Y = 0 \mid X = x) = 1 - \eta(x) & \text{si } g(x) = 1,\\[4pt]
\mathbb{P}(Y = 1 \mid X = x) = \eta(x) & \text{si } g(x) = 0.
\end{cases}$$

De façon compacte :
$$\mathbb{P}\big(g(X) \neq Y \mid X = x\big) = \mathbb{1}\{g(x)=1\}\,(1-\eta(x)) + \mathbb{1}\{g(x)=0\}\,\eta(x).$$

### Étape 3 — Le meilleur choix ponctuel est $g^\star$

En $x$, on choisit entre deux valeurs d'erreur possibles : $\eta(x)$ (si on prédit $0$) ou $1-\eta(x)$ (si on prédit $1$).
Le minimum des deux vaut par définition $\min(\eta(x),\,1-\eta(x))$, et il est atteint en prédisant la **classe la plus probable** :
$$g^\star(x) = \mathbb{1}\{\eta(x) \ge \tfrac{1}{2}\}
\quad\Longrightarrow\quad
\mathbb{P}\big(g^\star(X) \neq Y \mid X = x\big) = \min\big(\eta(x),\,1-\eta(x)\big).$$

*Vérification :* si $\eta(x) \ge 1/2$ alors $1-\eta(x) \le \eta(x)$, on prédit $1$ et l'erreur est $1-\eta(x) = \min$. Si $\eta(x) < 1/2$, on prédit $0$ et l'erreur est $\eta(x) = \min$. ✓

### Étape 4 — Optimalité et conclusion

Pour **tout** classifieur $g$ et tout $x$, on a par définition du minimum :
$$\mathbb{P}\big(g(X) \neq Y \mid X = x\big) \;\ge\; \min\big(\eta(x),\,1-\eta(x)\big) = \mathbb{P}\big(g^\star(X) \neq Y \mid X = x\big).$$
En prenant l'espérance sur $X$ (Étape 1), l'inégalité est **préservée** :
$$L(g) \;\ge\; \mathbb{E}\big[\min(\eta(X),\,1-\eta(X))\big] = L(g^\star) = L^\star.$$

Ainsi $g^\star$ minimise le risque, et son risque vaut exactement
$$\boxed{\,L^\star = \mathbb{E}\big[\min\big(\eta(X),\,1 - \eta(X)\big)\big].\,}$$

> **Remarque utile.** On peut aussi écrire $\min(\eta, 1-\eta) = \tfrac{1}{2} - \big|\eta - \tfrac{1}{2}\big|$, d'où
> $L^\star = \tfrac{1}{2} - \mathbb{E}\big|\eta(X) - \tfrac{1}{2}\big|$ : le risque est **minimal** quand $\eta$ est loin de $1/2$
> (classes séparées) et **maximal** ($= 1/2$) quand $\eta \equiv 1/2$ (bruit pur).

---

# 📎 ANNEXE C — Si le classifieur de Bayes minimise le risque, pourquoi n'est-il pas « le modèle le plus performant » ?

---

C'est une question apparemment paradoxale : $g^\star$ **est** le classifieur optimal (Annexe B), et pourtant on
ne peut pas dire que c'est « le meilleur modèle » au sens usuel. Trois raisons, à bien distinguer.

### 1. « Optimal » ne veut pas dire « parfait » — l'erreur $L^\star$ est incompressible

$g^\star$ atteint le **plus petit risque possible**, mais ce risque n'est en général **pas nul** :
$$L^\star = \mathbb{E}\big[\min(\eta(X), 1-\eta(X))\big] > 0 \quad \text{dès que } \eta(x) \notin \{0, 1\}.$$
Cette erreur vient du **bruit intrinsèque** du problème (les classes se recouvrent : au même $x$ on peut observer
$Y=0$ ou $Y=1$). C'est de l'**incertitude aléatoire** (aléa irréductible), pas un défaut du modèle. Exemple : au
Cas 2, même le classifieur parfait se trompe **30 %** du temps. Aucun modèle — aussi sophistiqué soit-il — ne peut
descendre sous $L^\star$. Donc « le plus performant » n'a jamais signifié « sans erreur ».

### 2. $g^\star$ n'est **pas un modèle** que l'on peut entraîner — c'est une référence théorique

$g^\star$ dépend de la vraie fonction $\eta(x) = \mathbb{P}(Y=1 \mid X=x)$, c'est-à-dire de la **loi jointe
$(X,Y)$**, qui est **inconnue** en pratique. On ne dispose que d'un **échantillon fini**. On construit donc un
estimateur $\hat{g}_n$ (k-NN, régression logistique, arbres…) qui **approche** $g^\star$. Son risque se décompose :
$$\underbrace{L(\hat{g}_n) - L^\star}_{\text{excès de risque} \,\ge\, 0} \;=\;
\underbrace{\text{biais d'approximation}}_{\text{la classe de modèles contient-elle } g^\star ?} \;+\;
\underbrace{\text{erreur d'estimation}}_{\text{données finies, variance}}.$$
$g^\star$ est la **cible idéale** (le « poteau d'arrivée ») servant à mesurer la qualité d'un modèle réel, mais on
ne peut **pas l'utiliser** directement : il n'est pas calculable à partir des données. C'est un **benchmark**, pas
un concurrent.

### 3. « Performant » dépend du **critère** — $g^\star$ optimise le coût 0–1, pas les autres

Le classifieur de Bayes avec seuil $1/2$ minimise le **taux d'erreur** (coût 0–1). Mais « performant » peut vouloir
dire autre chose selon le problème :
- **coûts asymétriques** (un faux négatif médical coûte plus cher qu'un faux positif) ⇒ le seuil optimal n'est
  plus $1/2$ mais dépend du rapport des coûts ;
- **classes déséquilibrées** ⇒ on regarde plutôt le rappel, la précision, le F1, la balanced accuracy… ;
- **problème de score/classement** (AUC-ROC) ⇒ on évalue l'ordonnancement, pas une décision binaire.

Pour ces critères, la règle « prédire la classe la plus probable » n'est **pas** celle qui optimise la métrique
visée. $g^\star$ est optimal **uniquement pour le risque 0–1** qui le définit.

### En résumé

| Affirmation | Vrai / Faux | Pourquoi |
|---|---|---|
| $g^\star$ minimise le risque 0–1 | ✅ Vrai | Démontré en Annexe B |
| $g^\star$ ne se trompe jamais | ❌ Faux | Erreur incompressible $L^\star > 0$ (bruit) |
| $g^\star$ est un modèle qu'on peut entraîner | ❌ Faux | Il dépend de $\eta$ inconnue ; c'est une référence théorique |
| $g^\star$ est le meilleur pour toute métrique | ❌ Faux | Optimal pour le coût 0–1 seulement (pas coûts asymétriques, F1, AUC…) |

**Conclusion.** Le classifieur de Bayes est le **meilleur possible pour le taux d'erreur**, ce qui en fait la borne
de référence — mais il reste **imparfait** (à cause du bruit), **inaccessible** (loi inconnue) et **spécifique à une
métrique**. « Meilleur classifieur théorique » ≠ « meilleur modèle utilisable en pratique ».